# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. The dataset covers protein abundance, modifications, and sequences from human mast cell extracellular vesicles.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs for reference.

In [ ]:
# List all record sets and their fields with @id references
print("Available record sets and their fields:\n")
record_sets = []
for rs in metadata.record_sets:
    print(f"  Record Set @id: {rs['@id']}  |  name: {rs.get('name', None)}")
    record_sets.append(rs['@id'])
    if hasattr(rs, 'fields') or 'fields' in rs:
        fields = rs['fields'] if 'fields' in rs else getattr(rs, 'fields')
        for field in fields:
            print(f"    - Field @id: {field['@id']}, name: {field.get('name', None)}, dataType: {field.get('dataType', None)}")
    print()

print(f"All RecordSet @ids: {record_sets}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Example: select the main protein record set
main_record_set_id = record_sets[0] if len(record_sets) else None
# Load all record sets into pandas DataFrames
dataframes = {}

if main_record_set_id is not None:
    for rs_id in record_sets:
        print(f"Loading records for record set: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)

    df = dataframes[main_record_set_id]
    print(f"\nColumns in main record set ({main_record_set_id}):\n{df.columns.tolist()}")
    display(df.head())
else:
    print("No record sets found in the dataset.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data.

In [ ]:
# Example field IDs (replace with actual @id from Data Overview: e.g. /frontiers/7154140/proteins/Protein-RecordSet)
# We'll infer the first numeric field from the DataFrame columns and use it for demonstration.
import numpy as np

if main_record_set_id is not None:
    df = dataframes[main_record_set_id]
    # Identify potential numeric fields from dtypes
    num_fields = [col for col in df.columns if np.issubdtype(df[col].dtype, np.number)]
    if num_fields:
        numeric_field = num_fields[0]
        print(f"Using numeric field for analysis: {numeric_field}")
    else:
        print("No numeric fields detected. EDA step will not run.")

    # Example: filter by threshold
    threshold = df[numeric_field].mean() if num_fields else 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (
        filtered_df[numeric_field] - filtered_df[numeric_field].mean()
    ) / filtered_df[numeric_field].std()

    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a likely categorical field
    object_fields = [col for col in df.columns if df[col].dtype == 'object']
    group_field = object_fields[0] if object_fields else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
        print(f"Grouped data by {group_field} (showing means):")
        display(grouped_df.head())
else:
    print("No main record set; EDA not performed.")

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id is not None and num_fields:
    # Histogram of the selected numeric field
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

    # If at least 2 numeric columns, scatter plot
    if len(num_fields) >= 2:
        plt.figure(figsize=(6, 6))
        sns.scatterplot(x=df[num_fields[0]], y=df[num_fields[1]])
        plt.title(f"{num_fields[0]} vs {num_fields[1]}")
        plt.xlabel(num_fields[0])
        plt.ylabel(num_fields[1])
        plt.show()
else:
    print("Visualization skipped: no numeric data found.")

## 6. Conclusion
This notebook demonstrated loading, exploring, processing, and visualizing the FAIR^2 mass spectrometry protein dataset using `mlcroissant`.

- **Data Access**: Flexible loading by referencing all entities via their `@id` as per the Croissant schema.
- **Exploration**: Inspected available record sets and fields. Loaded as pandas DataFrames for analysis.
- **EDA & Visualization**: Performed standard normalization and grouping, and visualized distributions.

You can extend this notebook to perform domain-specific or advanced analyses on the dataset fields most relevant to your research.